# RQ2 — Explainability (SHAP) of Wildfire Risk Predictions
*Which features drive the predictions, and do SHAP explanations match fire-weather domain knowledge?*

**Outputs:** `RQ2_shap_importance.csv`, `RQ2_shap_summary.pdf`

In [4]:

# ============================================================
# Shared data loading & preprocessing (Algerian Forest Fires)
# ============================================================
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True,
})

OUTDIR = "/kaggle/working"          # on Kaggle this is the output folder
os.makedirs(OUTDIR, exist_ok=True)

def find_csv():
    """Find the Algerian Forest Fires csv anywhere under /kaggle/input."""
    cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not cands:                   # local fallback
        cands = glob.glob("**/*.csv", recursive=True)
    # prefer a file whose name mentions 'algerian' or 'forest'
    for c in cands:
        n = os.path.basename(c).lower()
        if "algerian" in n or "forest" in n or "fire" in n:
            return c
    return cands[0]

def load_algerian():
    path = find_csv()
    print("Loading:", path)
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    # The raw UCI file mixes a title line, two region headers and a blank row,
    # so we parse it line by line rather than with a fixed-width csv reader.
    rows = [ln.split(",") for ln in lines]
    header = None
    region = "Bejaia"           # first block in the UCI file
    region_switched = False
    records, cols = [], None
    for r in rows:
        cells = [str(x).strip() for x in r]
        joined = " ".join(cells).lower()
        if "temperature" in joined and ("rh" in joined or "ws" in joined):
            cols = [c.strip() for c in cells if c.strip() != ""]
            header = cols
            if records and not region_switched:
                region = "Sidi-Bel Abbes"   # second header => second region
                region_switched = True
            continue
        if all(c == "" for c in cells):
            continue
        if "region" in joined or "dataset" in joined:   # title / region label line
            if "sidi" in joined:
                region = "Sidi-Bel Abbes"; region_switched = True
            continue
        if header is None:
            continue
        data_cells = [c for c in cells if c != ""]
        if len(data_cells) < len(cols):
            continue
        rec = dict(zip(cols, data_cells[:len(cols)]))
        rec["region"] = region
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    return df

df = load_algerian()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Standardise the target column name
target_col = [c for c in df.columns if "class" in c.lower()]
target_col = target_col[0] if target_col else df.columns[-2]
df = df.rename(columns={target_col: "Classes"})

# Clean target -> binary (fire = 1, not fire = 0)
df["Classes"] = (df["Classes"].astype(str).str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True))
df = df[df["Classes"].isin(["fire", "not fire"])].copy()
df["target"] = (df["Classes"] == "fire").astype(int)

# Numeric feature columns
FEATURES = ["Temperature", "RH", "Ws", "Rain",
            "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
FEATURES = [f for f in FEATURES if f in df.columns]
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

print("Shape:", df.shape, "| Fire:", int(df.target.sum()),
      "| Not fire:", int((1-df.target).sum()))
print("Regions:", df["region"].value_counts().to_dict())
X = df[FEATURES].copy()
y = df["target"].copy()


Loading: /kaggle/input/notebooks/sudhanshu432/eda-and-fe-algerian-forest-fires-dataset/Algerian_forest_fires_cleaned_dataset.csv
Shape: (243, 17) | Fire: 137 | Not fire: 106
Regions: {'Bejaia': 243}


In [5]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import shap

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=400, random_state=42).fit(X_tr, y_tr)

explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_te)
# Normalise across SHAP versions: may be list[class] or 3D array (n,feat,class)
sv = np.array(sv) if not isinstance(sv, list) else sv
if isinstance(sv, list):
    sv_pos = np.asarray(sv[1])
elif sv.ndim == 3:
    sv_pos = sv[:, :, 1]          # positive class
else:
    sv_pos = sv
sv_pos = np.asarray(sv_pos)
if sv_pos.shape != (X_te.shape[0], X_te.shape[1]):
    sv_pos = sv_pos.reshape(X_te.shape[0], X_te.shape[1])

mean_abs = np.abs(sv_pos).mean(axis=0)
imp = (pd.DataFrame({"Feature": X_te.columns, "Mean_abs_SHAP": mean_abs})
       .sort_values("Mean_abs_SHAP", ascending=False).round(4))
imp.to_csv(f"{OUTDIR}/RQ2_shap_importance.csv", index=False)
print(imp.to_string(index=False))


    Feature  Mean_abs_SHAP
       FFMC         0.1674
        ISI         0.1573
        FWI         0.0946
         DC         0.0299
        BUI         0.0147
        DMC         0.0140
       Rain         0.0123
         RH         0.0025
Temperature         0.0020
         Ws         0.0010


In [6]:

plt.figure()
shap.summary_plot(sv_pos, X_te, show=False, plot_size=(7, 5))
plt.title("RQ2: SHAP Summary — Feature Impact on Wildfire Risk")
plt.savefig(f"{OUTDIR}/RQ2_shap_summary.pdf", bbox_inches="tight")
plt.close()
print("Saved RQ2_shap_summary.pdf")


Saved RQ2_shap_summary.pdf
